In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# === Load data ===
df = pd.read_csv(r"G:\Hangkai\CONUS_Forest_Edge_LCMAP\key_outputs\Forest_Depth_Change_historical_Disturbance_Attribution_1988_2021_pa&ownership.csv")

# === Add region & PA labels ===
state_to_region = {
    'Connecticut': 'Northeast', 'Maine': 'Northeast', 'Massachusetts': 'Northeast',
    'New Hampshire': 'Northeast', 'Rhode Island': 'Northeast', 'Vermont': 'Northeast',
    'New Jersey': 'Northeast', 'New York': 'Northeast', 'Pennsylvania': 'Northeast',
    'Illinois': 'Midwest', 'Indiana': 'Midwest', 'Michigan': 'Midwest',
    'Ohio': 'Midwest', 'Wisconsin': 'Midwest', 'Iowa': 'Midwest', 'Kansas': 'Midwest',
    'Minnesota': 'Midwest', 'Missouri': 'Midwest', 'Nebraska': 'Midwest',
    'North Dakota': 'Midwest', 'South Dakota': 'Midwest', 'Delaware': 'South',
    'District of Columbia': 'South', 'Florida': 'South', 'Georgia': 'South',
    'Maryland': 'South', 'North Carolina': 'South', 'South Carolina': 'South',
    'Virginia': 'South', 'West Virginia': 'South', 'Alabama': 'South',
    'Kentucky': 'South', 'Mississippi': 'South', 'Tennessee': 'South',
    'Arkansas': 'South', 'Louisiana': 'South', 'Oklahoma': 'South', 'Texas': 'South',
    'Arizona': 'West', 'Colorado': 'West', 'Idaho': 'West', 'Montana': 'West',
    'Nevada': 'West', 'New Mexico': 'West', 'Utah': 'West', 'Wyoming': 'West',
    'Alaska': 'West', 'California': 'West', 'Hawaii': 'West', 'Oregon': 'West',
    'Washington': 'West'
}
df["Region"] = df["Ecoregion"].map(state_to_region)
df["PA_Label"] = df["ProtectedAreaClass"].apply(lambda x: "Protected Area" if x > 0 else "Non-Protected")
df["From"] = df["ForestChangeType"] // 10
df["To"] = df["ForestChangeType"] % 10

# === Filter for Fragmentation (interior → edge) ===
fragmented = df[
    ((df["From"] == 5) & (df["To"].isin([1, 2, 3, 4]))) |
    ((df["From"] == 4) & (df["To"].isin([1, 2, 3]))) |
    ((df["From"] == 3) & (df["To"].isin([1, 2]))) |
    ((df["From"] == 2) & (df["To"] == 1))
].copy()

# === Compute total forest pixels for normalization ===
total_forest = df[df["From"].isin([1, 2, 3, 4, 5])]
total_forest = total_forest.groupby(["Region", "PA_Label", "Year_From"])["PixelCount"].sum().reset_index()
total_forest = total_forest.rename(columns={"PixelCount": "TotalForestPixels"})

# === Merge and compute fragmentation percentage ===
fragmented = fragmented.merge(total_forest, on=["Region", "PA_Label", "Year_From"], how="left")
fragmented["Percentage"] = fragmented["PixelCount"] / fragmented["TotalForestPixels"] * 100

# === Aggregate by Disturbance Category ===
agg_df = fragmented.groupby(
    ["Region", "PA_Label", "DisturbanceCategory", "Year_From"]
)["Percentage"].sum().reset_index()

# === Disturbance categories & colors ===
disturbance_labels = [
    "No Disturbance Detected", "Logging", "Construction", "Stress",
    "Natural Hazard", "Water Dynamic", "Fire", "Agriculture Activity", "Other"
]
disturbance_colors = [
    'black', '#1b9e77', 'purple', '#e7298a',
    '#66a61e', '#1f78b4', 'red', 'gold', 'gray'
]
color_map = dict(zip(disturbance_labels, disturbance_colors))

In [ ]:
df

In [ ]:
import pandas as pd

# === Load data ===
df = pd.read_csv(r"G:\Hangkai\CONUS_Forest_Edge_LCMAP\key_outputs\Forest_Depth_Change_historical_Disturbance_Attribution_1988_2021_pa&ownership.csv")

# === Add region labels ===
state_to_region = {
    'Connecticut': 'Northeast', 'Maine': 'Northeast', 'Massachusetts': 'Northeast',
    'New Hampshire': 'Northeast', 'Rhode Island': 'Northeast', 'Vermont': 'Northeast',
    'New Jersey': 'Northeast', 'New York': 'Northeast', 'Pennsylvania': 'Northeast',
    'Illinois': 'Midwest', 'Indiana': 'Midwest', 'Michigan': 'Midwest',
    'Ohio': 'Midwest', 'Wisconsin': 'Midwest', 'Iowa': 'Midwest', 'Kansas': 'Midwest',
    'Minnesota': 'Midwest', 'Missouri': 'Midwest', 'Nebraska': 'Midwest',
    'North Dakota': 'Midwest', 'South Dakota': 'Midwest', 'Delaware': 'South',
    'District of Columbia': 'South', 'Florida': 'South', 'Georgia': 'South',
    'Maryland': 'South', 'North Carolina': 'South', 'South Carolina': 'South',
    'Virginia': 'South', 'West Virginia': 'South', 'Alabama': 'South',
    'Kentucky': 'South', 'Mississippi': 'South', 'Tennessee': 'South',
    'Arkansas': 'South', 'Louisiana': 'South', 'Oklahoma': 'South', 'Texas': 'South',
    'Arizona': 'West', 'Colorado': 'West', 'Idaho': 'West', 'Montana': 'West',
    'Nevada': 'West', 'New Mexico': 'West', 'Utah': 'West', 'Wyoming': 'West',
    'Alaska': 'West', 'California': 'West', 'Hawaii': 'West', 'Oregon': 'West',
    'Washington': 'West'
}
df["Region"] = df["Ecoregion"].map(state_to_region)
df["From"] = df["ForestChangeType"] // 10
df["To"] = df["ForestChangeType"] % 10

# === Filter for fragmentation transitions: interior → edge ===
fragmented = df[
    ((df["From"] == 5) & (df["To"].isin([1, 2, 3, 4]))) |
    ((df["From"] == 4) & (df["To"].isin([1, 2, 3]))) |
    ((df["From"] == 3) & (df["To"].isin([1, 2]))) |
    ((df["From"] == 2) & (df["To"] == 1))
].copy()

# === Only consider fragmentation from depth 5 to edge (1–4) ===
EFT = df[(df["From"] == 5) & (df["To"].isin([1, 2, 3, 4]))].copy()

# === Total forest loss (any forest to non-forest) per region/disturbance/year ===
loss = df[df["To"] == 0].groupby(
    ["Region", "DisturbanceCategory", "Year_From"]
)["PixelCount"].sum().reset_index()
loss = loss.rename(columns={"PixelCount": "TotalLossPixels"})

# === Total fragmentation (interior → exterior) per region/disturbance/year ===
frag = EFT.groupby(
    ["Region", "DisturbanceCategory", "Year_From"]
)["PixelCount"].sum().reset_index()
frag = frag.rename(columns={"PixelCount": "FragmentedPixels"})

# === Merge and calculate EFTR ===
eftr_df = pd.merge(frag, loss, on=["Region", "DisturbanceCategory", "Year_From"], how="left")
eftr_df["EFTR"] = eftr_df["FragmentedPixels"] / eftr_df["TotalLossPixels"]

# === Optional: Pivot into matrix (Region x DisturbanceCategory, with EFTR values per year) ===
eftr_matrix = eftr_df.pivot_table(
    index=["Region", "DisturbanceCategory"],
    columns="Year_From",
    values="EFTR"
)

In [ ]:
eftr_matrix

In [ ]:
import pandas as pd
from pymannkendall import original_test as mk_test

# === Load EFTR result ===
# Assume you already have `eftr_df` as computed earlier:
# Columns: ['Region', 'DisturbanceCategory', 'Year_From', 'FragmentedPixels', 'TotalLossPixels', 'EFTR']

# === 1. Regional EFTR per year ===
regional_eftr = eftr_df.groupby(["Region", "Year_From"]).agg({
    "FragmentedPixels": "sum",
    "TotalLossPixels": "sum"
}).reset_index()
regional_eftr["EFTR"] = regional_eftr["FragmentedPixels"] / regional_eftr["TotalLossPixels"]

# === 2. Whole CONUS EFTR per year ===
conus_eftr = eftr_df.groupby("Year_From").agg({
    "FragmentedPixels": "sum",
    "TotalLossPixels": "sum"
}).reset_index()
conus_eftr["Region"] = "CONUS"
conus_eftr["EFTR"] = conus_eftr["FragmentedPixels"] / conus_eftr["TotalLossPixels"]

# === 3. Combine for full trend analysis ===
combined_eftr = pd.concat([regional_eftr[["Region", "Year_From", "EFTR"]], conus_eftr[["Region", "Year_From", "EFTR"]]])

# === 4. Mann-Kendall trend test by Region ===
trend_results = []
for region, group in combined_eftr.groupby("Region"):
    result = mk_test(group["EFTR"].values)
    trend_results.append({
        "Region": region,
        "Trend": result.trend,
        "p-value": result.p,
        "Tau": result.Tau,
        "Slope": result.slope
    })
trend_df = pd.DataFrame(trend_results)

# Print top 10 rows of trend results
print("=== Mann-Kendall EFTR Trend Results ===")
print(trend_df.head(10))

In [ ]:
# === Compute mean EFTR per region
mean_eftr = combined_eftr.groupby("Region")["EFTR"].mean().reset_index()
mean_eftr = mean_eftr.rename(columns={"EFTR": "Mean_EFTR"})

# === Merge with Mann-Kendall trend result
trend_summary = pd.merge(trend_df, mean_eftr, on="Region")

# === Reorder for clarity
trend_summary = trend_summary[["Region", "Mean_EFTR", "Trend", "Tau", "Slope", "p-value"]]

# === Display or save
print("=== EFTR Mean and Trend Summary ===")
print(trend_summary)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# === Set up plotting style ===
sns.set(style="whitegrid")
plt.figure(figsize=(10, 6))

# === Plot all regions (including CONUS) ===
regions_to_plot = combined_eftr["Region"].unique()
palette = sns.color_palette("tab10", len(regions_to_plot))

# === Plot line for each region ===
for idx, region in enumerate(regions_to_plot):
    subset = combined_eftr[combined_eftr["Region"] == region]
    plt.plot(subset["Year_From"], subset["EFTR"], marker='o', label=region, color=palette[idx])

# === Labels and legend ===
plt.title("EFTR Trend Over Time by Region")
plt.xlabel("Year")
plt.ylabel("EFTR (Exterior Forest Transition Ratio)")
plt.ylim(0, combined_eftr["EFTR"].max() * 1.1)
plt.legend(title="Region", loc="upper left")
plt.tight_layout()
plt.grid(True)

# === Show or save ===
plt.savefig("EFTR_trend_by_region.png", dpi=300)
plt.show()

In [ ]:
import pandas as pd
from pymannkendall import original_test as mk_test

# === Add CONUS row by aggregating eftr_df ===
conus_df = eftr_df.groupby(["Year_From"]).agg({
    "FragmentedPixels": "sum",
    "TotalLossPixels": "sum"
}).reset_index()
conus_df["Region"] = "CONUS"
conus_df["EFTR"] = conus_df["FragmentedPixels"] / conus_df["TotalLossPixels"]

# === Combine CONUS and regional EFTR ===
regional_eftr = eftr_df.groupby(["Region", "Year_From"]).agg({
    "FragmentedPixels": "sum",
    "TotalLossPixels": "sum"
}).reset_index()
regional_eftr["EFTR"] = regional_eftr["FragmentedPixels"] / regional_eftr["TotalLossPixels"]

combined = pd.concat([
    regional_eftr[["Region", "Year_From", "FragmentedPixels", "TotalLossPixels", "EFTR"]],
    conus_df[["Region", "Year_From", "FragmentedPixels", "TotalLossPixels", "EFTR"]]
])

# === Initialize results list ===
results = []

for region, group in combined.groupby("Region"):
    # Yearly average EFTR
    mean_eftr_year_avg = group["EFTR"].mean()
    
    # Total FragmentedPixels / TotalLossPixels
    mean_eftr_sum_first = group["FragmentedPixels"].sum() / group["TotalLossPixels"].sum()
    
    # Mann-Kendall test
    mk = mk_test(group["EFTR"].values)
    
    results.append({
        "Region": region,
        "Mean_EFTR_YearAvg": mean_eftr_year_avg,
        "Mean_EFTR_SumFirst": mean_eftr_sum_first,
        "Sen_Slope": mk.slope,
        "Tau": mk.Tau,
        "p_value": mk.p
    })

# === Create DataFrame and round ===
summary_df = pd.DataFrame(results)
summary_df = summary_df

# === Display or save ===
print("=== EFTR Summary Table ===")
summary_df